In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# Load data
train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

# Combine so ticket-group counts are accurate across both files
train['is_train'] = 1
test['is_train'] = 0
test['Survived'] = np.nan
combined = pd.concat([train, test], sort=False)

# --- Ticket group size + bucket ---
combined['TicketGroupSize'] = combined.groupby('Ticket')['Ticket'].transform('count')

def bucket_group(size):
    if size == 1:
        return 'Solo'
    elif size <= 3:
        return 'Small'
    elif size == 4:
        return 'Medium'
    else:
        return 'Large'

combined['TicketGroup'] = combined['TicketGroupSize'].apply(bucket_group)

# --- Feature engineering ---
def engineer(df):
    df = df.copy()
    df['Title'] = df['Name'].str.extract(r',\s*([^\.]*)\.')
    df['Title'] = df['Title'].str.replace('the Countess', 'Countess', regex=False)
    rare = ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona']
    df['Title'] = df['Title'].replace(rare, 'Rare')
    df['Title'] = df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    return df

combined = engineer(combined)

# Impute Age using the median within each Title group (fit on train rows only)
title_age_median = combined[combined['is_train'] == 1].groupby('Title')['Age'].median()
global_median = combined[combined['is_train'] == 1]['Age'].median()
combined['Age'] = combined.apply(
    lambda r: title_age_median.get(r['Title'], global_median) if pd.isnull(r['Age']) else r['Age'],
    axis=1
)

# --- Encode categoricals ---
combined['Sex'] = combined['Sex'].map({'male': 0, 'female': 1})
combined['Embarked'] = combined['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
combined['Title'] = combined['Title'].map({'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4})
combined['TicketGroup'] = combined['TicketGroup'].map({'Solo': 0, 'Small': 1, 'Medium': 2, 'Large': 3})

# --- Split back into train/test ---
train_p = combined[combined['is_train'] == 1].copy()
test_p = combined[combined['is_train'] == 0].copy()

features = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone',
            'TicketGroupSize', 'TicketGroup']

X = train_p[features]
y = train_p['Survived'].astype(int)
X_test = test_p[features]

# --- Train: shallow, fixed-parameter RF (no grid search, to avoid overfitting to CV folds) ---
rf = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=5, random_state=42)
rf.fit(X, y)

# --- Predict and save submission ---
test_preds = rf.predict(X_test)
submission = pd.DataFrame({'PassengerId': test_p['PassengerId'].astype(int), 'Survived': test_preds})
submission.to_csv('submission.csv', index=False)